#### imports


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report

#### load the diagnosis final result data

In [3]:
df = pd.read_csv("../data/processed/diagnosis_final.csv")

df.head()

,age,sex,pregnant,location,travel_history_endemic_area,bed_net_use,irs_spraying,previous_malaria_episodes,fever,chills_rigors,...,altered_consciousness,seizures,hemoglobin_g_dl,platelets_x10e9_l,wbc_x10e9_l,glucose_mg_dl,creatinine_mg_dl,bilirubin_mg_dl,lactate_mmol_l,diagnosis
0,15,F,False,Sub-Saharan Africa - Rural,False,False,True,3,True,True,...,False,False,10.6,132,5.0,106,0.34,1.5,1.8,0
1,27,F,False,Latin America - Urban,False,False,False,0,False,False,...,False,False,14.1,360,8.7,80,0.73,1.0,1.8,1
2,21,F,False,Papua New Guinea,False,True,False,2,False,True,...,False,False,10.1,268,3.8,105,0.76,1.1,1.9,0
3,17,F,False,Southeast Asia - Rural,False,True,True,2,False,False,...,False,False,13.6,216,9.2,99,0.40,0.5,2.0,1
4,13,F,False,Sub-Saharan Africa - Urban,True,True,False,4,True,True,...,False,False,13.5,128,5.1,86,0.47,1.4,0.8,0


#### split te data

In [4]:
X = df.drop("diagnosis", axis=1)
y = df["diagnosis"]

#### train test split

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape)
print(X_test.shape)

(4000, 27)
(1000, 27)


#### identify columns

In [6]:
X.dtypes

age                              int64
sex                                str
pregnant                          bool
location                           str
travel_history_endemic_area       bool
bed_net_use                       bool
irs_spraying                      bool
previous_malaria_episodes        int64
fever                             bool
chills_rigors                     bool
headache                          bool
night_sweats                      bool
fatigue_malaise                   bool
nausea_vomiting                   bool
diarrhea                          bool
cough                             bool
abdominal_pain                    bool
jaundice                          bool
altered_consciousness             bool
seizures                          bool
hemoglobin_g_dl                float64
platelets_x10e9_l                int64
wbc_x10e9_l                    float64
glucose_mg_dl                    int64
creatinine_mg_dl               float64
bilirubin_mg_dl          

In [7]:
categorical_cols = [
    "sex",
    "location"
]

In [8]:
numeric_cols = [
    col for col in X.columns
    if col not in categorical_cols
]

#### create preprocessor

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ],
    remainder="passthrough"
)

In [10]:
print(preprocessor)

ColumnTransformer(remainder='passthrough',
                  transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['sex', 'location'])])


#### pipeline

In [11]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    (
        "classifier",
        RandomForestClassifier(
            random_state=42
        )
    )
])

In [12]:
rf_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the o

In [14]:
print(X_train.shape)
print(X_test.shape)
print(X.columns.tolist())

(4000, 27)
(1000, 27)
['age', 'sex', 'pregnant', 'location', 'travel_history_endemic_area', 'bed_net_use', 'irs_spraying', 'previous_malaria_episodes', 'fever', 'chills_rigors', 'headache', 'night_sweats', 'fatigue_malaise', 'nausea_vomiting', 'diarrhea', 'cough', 'abdominal_pain', 'jaundice', 'altered_consciousness', 'seizures', 'hemoglobin_g_dl', 'platelets_x10e9_l', 'wbc_x10e9_l', 'glucose_mg_dl', 'creatinine_mg_dl', 'bilirubin_mg_dl', 'lactate_mmol_l']


#### create random forest baseline

In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        random_state=42
    ))
])

baseline_model.fit(X_train, y_train)

y_pred = baseline_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))

Accuracy: 0.993
Precision: 0.9900142653352354
Recall: 1.0
F1: 0.9949820788530466


## Hyperparameter Tuning
#### parameter grid

In [16]:
param_grid = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [None, 5, 10, 20, 30],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4],
    "classifier__max_features": ["sqrt", "log2"]
}

#### random search

In [17]:
random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="f1",
    random_state=42,
    n_jobs=-1,
    verbose=2
)

#### train

In [18]:
random_search.fit(X_train, y_train)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__max_depth': [None, 5, ...], 'classifier__max_features': ['sqrt', 'log2'], 'classifier__min_samples_leaf': [1, 2, ...], 'classifier__min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimat

#### best parameter

In [19]:
print("Best Parameters:")
print(random_search.best_params_)

print("\nBest CV Score:")
print(random_search.best_score_)

Best Parameters:
{'classifier__n_estimators': 300, 'classifier__min_samples_split': 5, 'classifier__min_samples_leaf': 1, 'classifier__max_features': 'log2', 'classifier__max_depth': None}

Best CV Score:
0.9947949654098152


#### evaluate tunned model

In [20]:
best_model = random_search.best_estimator_

y_pred_tuned = best_model.predict(X_test)

print(classification_report(
    y_test,
    y_pred_tuned
))

              precision    recall  f1-score   support

           0       1.00      0.98      0.99       306
           1       0.99      1.00      1.00       694

    accuracy                           0.99      1000
   macro avg       1.00      0.99      0.99      1000
weighted avg       0.99      0.99      0.99      1000



#### create comparison table

In [21]:
comparison = pd.DataFrame({
    "Model": ["Baseline RF", "Tuned RF"],
    "Accuracy": [
        accuracy_score(y_test, y_pred),
        accuracy_score(y_test, y_pred_tuned)
    ],
    "F1": [
        f1_score(y_test, y_pred),
        f1_score(y_test, y_pred_tuned)
    ]
})

comparison

,Model,Accuracy,F1
0,Baseline RF,0.993,0.994982
1,Tuned RF,0.994,0.995696


In [22]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    best_model,
    "../models/malaria_diagnosis_rf.pkl"
)

['../models/malaria_diagnosis_rf.pkl']